In [1]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import models
import matplotlib.pyplot as plt

IMG_SIZE = 224
BATCH_SIZE = 32

DATASET_PATH = "dataset"

In [2]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

print(train_ds.class_names)

Found 42330 files belonging to 4 classes.
Using 33864 files for training.
Found 42330 files belonging to 4 classes.
Using 8466 files for validation.
['COVID', 'LUNG_OPACITY', 'NORMAL', 'PNEUMONIA']


In [3]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

In [4]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

In [5]:
model = models.Sequential([
    layers.Rescaling(1./255),

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dropout(0.3),

    layers.Dense(128, activation='relu'),

    layers.Dense(4, activation='softmax')
])

In [6]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ mobilenetv2_1.00_224 (Functional)    │ (None, 7, 7, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ ?                           │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,257,984 (8.61 MB)

In [7]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
1059/1059 ━━━━━━━━━━━━━━━━━━━━ 608s 570ms/step - accuracy: 0.7342 - loss: 0.6722 - val_accuracy: 0.7738 - val_loss: 0.5650
Epoch 2/10
1059/1059 ━━━━━━━━━━━━━━━━━━━━ 1071s 994ms/step - accuracy: 0.7736 - loss: 0.5723 - val_accuracy: 0.8045 - val_loss: 0.5119
Epoch 3/10
1059/1059 ━━━━━━━━━━━━━━━━━━━━ 1135s 1s/step - accuracy: 0.7867 - loss: 0.5394 - val_accuracy: 0.8096 - val_loss: 0.4893
Epoch 4/10
1059/1059 ━━━━━━━━━━━━━━━━━━━━ 1141s 1s/step - accuracy: 0.7932 - loss: 0.5231 - val_accuracy: 0.8156 - val_loss: 0.4724
Epoch 5/10
1059/1059 ━━━━━━━━━━━━━━━━━━━━ 762s 685ms/step - accuracy: 0.7981 - loss: 0.5078 - val_accuracy: 0.8142 - val_loss: 0.4748
Epoch 6/10
1059/1059 ━━━━━━━━━━━━━━━━━━━━ 553s 522ms/step - accuracy: 0.8061 - loss: 0.4855 - val_accuracy: 0.8213 - val_loss: 0.4570
Epoch 8/10
1059/1059 ━━━━━━━━━━━━━━━━━━━━ 940s 888ms/step - accuracy: 0.8129 - loss: 0.4726 - val_accuracy: 0.8227 - val_loss: 0.4581
Epoch 9/10
1059/1059 ━━━━━━━━━━━━━━━━━━━━ 1147s 1s/step - accurac

In [8]:
model.save("modelo_covid_v2.keras")

In [9]:
model.save("modelo_covid_v2.keras")

In [10]:
from sklearn.metrics import classification_report
import numpy as np

y_true = []
y_pred = []

for images, labels in val_ds:
    pred = model.predict(images, verbose=0)

    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(pred, axis=1))

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.72      0.67      0.69      1470
           1       0.78      0.82      0.80      2449
           2       0.86      0.87      0.86      4048
           3       0.95      0.85      0.90       499

    accuracy                           0.82      8466
   macro avg       0.83      0.80      0.81      8466
weighted avg       0.82      0.82      0.82      8466



In [11]:
model.save("modelo_covid_v2.keras")

In [12]:
print(train_ds.class_names)

AttributeError: '_PrefetchDataset' object has no attribute 'class_names'

In [13]:
print(val_ds.class_names)

AttributeError: '_PrefetchDataset' object has no attribute 'class_names'

In [14]:
import os
print(sorted(os.listdir("dataset")))

['COVID', 'COVID.metadata.xlsx', 'LUNG_OPACITY', 'LUNG_OPACITY.metadata.xlsx', 'NORMAL', 'NORMAL.metadata.xlsx', 'PNEUMONIA', 'PNEUMONIA.metadata.xlsx', 'README.md.txt']


In [15]:
for images, labels in val_ds.take(1):
    print(labels.numpy()[:20])

[2 2 0 1 3 2 2 2 2 2 0 1 0 0 1 2 1 2 2 1]


In [16]:
print(model.output_shape)

(None, 4)


In [17]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.show()

ModuleNotFoundError: No module named 'seaborn'

In [1]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.show()


NameError: name 'y_true' is not defined

In [2]:
from sklearn.metrics import classification_report
import numpy as np

y_true = []
y_pred = []

for images, labels in val_ds:
    pred = model.predict(images, verbose=0)

    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(pred, axis=1))

print(classification_report(y_true, y_pred))

NameError: name 'val_ds' is not defined

In [3]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.show()

ValueError: zero-size array to reduction operation fmin which has no identity

<Figure size 800x600 with 0 Axes>